# 00 · Análise Exploratória de Dados (EDA)
Exploração das 3 abas da base de RDR e 1ª instância para entender qualidade, cobertura e distribuições antes da modelagem.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('Bibliotecas carregadas.')

## 1 · Carregar as 3 abas do Excel + CSV

In [ ]:
EXCEL = '../Base RDR e primeira instancia [incoming e julgadas].xlsx'
CSV   = '../Base RDR e primeira instancia [incoming e julgadas] - Histórico de 1ª Instância (Canais MP).csv'

aba1_excel = pd.read_excel(EXCEL, sheet_name=0)
aba2       = pd.read_excel(EXCEL, sheet_name=1)
aba3       = pd.read_excel(EXCEL, sheet_name=2)
aba1_csv   = pd.read_csv(CSV)

print(f'Aba 1 (Excel): {aba1_excel.shape}')
print(f'Aba 1 (CSV):   {aba1_csv.shape}')
print(f'Aba 2 (RDR):   {aba2.shape}')
print(f'Aba 3 (Mercado): {aba3.shape}')

## 2 · Limpeza e tipagem

In [ ]:
# Aba 1 — usar o CSV (mais limpo)
aba1 = aba1_csv.copy()
aba1.columns = aba1.columns.str.strip()
aba1['INCOMING_DTTM'] = pd.to_datetime(aba1['INCOMING_DTTM'], format='mixed')

# Aba 2
aba2.columns = aba2.columns.str.strip()
aba2['RDR_EXTERNAL_CREATE_DTTM'] = pd.to_datetime(aba2['RDR_EXTERNAL_CREATE_DTTM'], format='mixed')
aba2['RDR_OUTGOING_DTTM']        = pd.to_datetime(aba2['RDR_OUTGOING_DTTM'], format='mixed', errors='coerce')

print('Aba 1 — tipos:')
print(aba1.dtypes)
print()
print('Aba 2 — tipos:')
print(aba2.dtypes)

## 3 · Missing values

In [ ]:
print('=== Aba 1 ===')
print(aba1.isnull().sum())
print()
print('=== Aba 2 ===')
print(aba2.isnull().sum())

## 4 · Distribuição temporal — 1ª instância

In [ ]:
mensal_aba1 = (
    aba1.set_index('INCOMING_DTTM')
    .resample('ME')
    .size()
    .reset_index(name='contatos')
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(mensal_aba1['INCOMING_DTTM'].dt.to_period('M').astype(str),
       mensal_aba1['contatos'], color='steelblue')
ax.set_title('Volume mensal de acionamentos — 1ª Instância', fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Contatos')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../outputs/00_aba1_mensal.png', bbox_inches='tight')
plt.show()
print(mensal_aba1.to_string(index=False))

## 5 · Distribuição de status — Aba 2 (RDR)

In [ ]:
status_counts = aba2['RDR_STATUS_NAME'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(status_counts.index[::-1], status_counts.values[::-1],
               color=['#d62728' if 'procedente' in s.lower() and 'im' not in s.lower()
                      else '#ff7f0e' if 'pendente' in s.lower()
                      else '#2ca02c' for s in status_counts.index[::-1]])
ax.bar_label(ax.containers[0], padding=4)
ax.set_title('Distribuição de Status — Base RDR (Aba 2)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/00_status_rdr.png', bbox_inches='tight')
plt.show()
print(status_counts)

## 6 · Top canais (CX_TEAM_NAME)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_ch1 = aba1['CX_TEAM_NAME'].value_counts().head(10)
axes[0].barh(top_ch1.index[::-1], top_ch1.values[::-1], color='steelblue')
axes[0].bar_label(axes[0].containers[0], padding=4, fontsize=9)
axes[0].set_title('Top Canais — 1ª Instância', fontweight='bold')
axes[0].tick_params(axis='y', labelsize=8)

top_ch2 = aba2['CX_TEAM_NAME'].value_counts().head(10)
axes[1].barh(top_ch2.index[::-1], top_ch2.values[::-1], color='tomato')
axes[1].bar_label(axes[1].containers[0], padding=4, fontsize=9)
axes[1].set_title('Top Canais — RDR (Aba 2)', fontweight='bold')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('../outputs/00_canais.png', bbox_inches='tight')
plt.show()

## 7 · Top causas e produtos — Aba 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_cdu = aba2['CDU_NAME'].value_counts().head(12)
axes[0].barh(top_cdu.index[::-1], top_cdu.values[::-1], color='#9467bd')
axes[0].bar_label(axes[0].containers[0], padding=4, fontsize=8)
axes[0].set_title('Top CDU_NAME (Causa do Caso)', fontweight='bold')
axes[0].tick_params(axis='y', labelsize=8)

top_pr = aba2['CX_PR_NAME'].value_counts().head(12)
axes[1].barh(top_pr.index[::-1], top_pr.values[::-1], color='#8c564b')
axes[1].bar_label(axes[1].containers[0], padding=4, fontsize=8)
axes[1].set_title('Top CX_PR_NAME (Produto/Processo)', fontweight='bold')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('../outputs/00_causas_produtos.png', bbox_inches='tight')
plt.show()

## 8 · Cobertura do JOIN entre Aba 1 e Aba 2

In [ ]:
# Período sobreponível: set–nov 2025
aba1_set_nov = aba1[aba1['INCOMING_DTTM'] >= '2025-09-01'].copy()
aba1_set_nov['ID do usuário'] = pd.to_numeric(aba1_set_nov['ID do usuário'], errors='coerce')
aba2['ID do usuario']         = pd.to_numeric(aba2['ID do usuario'], errors='coerce')

ids_aba1 = set(aba1_set_nov['ID do usuário'].dropna().astype(int))
ids_aba2 = set(aba2['ID do usuario'].dropna().astype(int))

overlap = ids_aba1 & ids_aba2
print(f'IDs únicos Aba 1 (set-nov): {len(ids_aba1):,}')
print(f'IDs únicos Aba 2:           {len(ids_aba2):,}')
print(f'IDs em ambas as abas:       {len(overlap):,}')
print(f'Cobertura do JOIN:          {len(overlap)/len(ids_aba1)*100:.1f}%')

## 9 · Tabela de concorrência — Aba 3

In [ ]:
print('Aba 3 — Dados de Mercado:')
print(aba3.to_string())

## 10 · Exportar bases limpas

In [ ]:
aba1.to_csv('../outputs/aba1_limpa.csv', index=False)
aba2.to_csv('../outputs/aba2_limpa.csv', index=False)
aba3.to_csv('../outputs/aba3_limpa.csv', index=False)
print('Bases exportadas para ../outputs/')
print(f'  aba1_limpa.csv: {len(aba1):,} linhas')
print(f'  aba2_limpa.csv: {len(aba2):,} linhas')
print(f'  aba3_limpa.csv: {len(aba3):,} linhas')